# BLaVe-CoT — Demo LoRA BLIP-2 (luồng VS Code + Colab extension)

Notebook này chạy bản demo nhỏ (~118 mẫu, 2 epoch, QLoRA 4-bit) trên GPU T4 Colab,
chỉ để **kiểm chứng pipeline chạy thông** trước khi train đầy đủ trên GPU riêng.

## Cách dùng với extension "Google Colab" của VS Code
1. Cài extension **Google Colab** (publisher: Google) từ Marketplace.
2. Đăng nhập Google trong VS Code (Command Palette → `Colab: Sign in`).
3. Mở file `.ipynb` này, click selector kernel góc phải trên → **Connect to a Colab runtime** → chọn **T4 GPU**.
4. Chạy lần lượt các cell bên dưới.

Lưu ý: kernel + filesystem chạy ở backend Colab, không phải máy local. File `demo_pack.zip` ở máy bạn cần upload qua `files.upload()` ở cell 2.

## 1. Kiểm tra GPU + clone repo

Sau khi đã connect runtime T4, chạy cell sau để xác nhận GPU và kéo code từ GitHub.

In [3]:
import torch, os
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Clone code (nếu chưa có)
if not os.path.isdir('blave_train'):
    !git clone https://github.com/HuynhKhoaIT/blave_train.git
%cd blave_train
!ls *.py

CUDA: True | Tesla T4
/content/blave_train
config.py  inspect_model.py  model.py	      train.py
infer.py   make_demo.py      prepare_data.py


## 2. Cài lib + lấy `demo_pack.zip` từ Google Drive

**Trước khi chạy cell dưới:** upload `demo_pack.zip` (~50MB) lên Google Drive của bạn, đặt ở `MyDrive/blave_train/demo_pack.zip` (tạo folder `blave_train` trong MyDrive nếu chưa có).

⚠ Lưu ý: extension VS Code không hỗ trợ widget `files.upload()` (chỉ chạy trên Colab browser). Phải dùng Drive thay thế.

In [4]:
# Cài lib (lần đầu mất ~1-2 phút)
!pip install -q -r requirements.txt

# Mount Google Drive — popup yêu cầu quyền, accept để tiếp tục
from google.colab import drive
drive.mount('/content/drive')

# Copy demo_pack.zip từ Drive về runtime + giải nén
!cp /content/drive/MyDrive/blave_train/demo_pack.zip .
!unzip -q -o demo_pack.zip
!ls data/ && ls data/vizwiz/train | head -3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_converted.json  vizwiz
VizWiz_train_00000000.jpg
VizWiz_train_00000001.jpg
VizWiz_train_00000002.jpg


## 3. Verify config

Xác nhận PROFILE='demo' và đường dẫn khớp với data vừa unzip.

In [5]:
!python config.py

PROFILE đang chọn: demo

{
  "use_qlora": false,
  "batch_size": 2,
  "grad_accum": 2,
  "num_epochs": 2,
  "max_train_samples": 200,
  "lr": 0.0001,
  "save_every": 1,
  "output_dir": "./blip2_lora_demo",
  "profile_name": "demo",
  "base_model": "Salesforce/blip2-opt-2.7b",
  "paths": {
    "train_json": "data/train_converted.json",
    "val_json": "data/val_converted.json",
    "image_dir": "data/vizwiz/train",
    "val_image_dir": "data/vizwiz/val"
  },
  "lora": {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": [
      "query",
      "key"
    ]
  },
  "max_prompt_len": 32,
  "max_answer_len": 16
}


## 4. Train

Lần đầu sẽ tải BLIP-2 base (~10GB) từ HuggingFace — mất ~3-5 phút. Sau đó train 2 epoch trên 118 mẫu, QLoRA 4-bit → khoảng 3-5 phút trên T4.

Nếu báo `target modules not found`: chạy `!python inspect_model.py` rồi sửa `LORA["target_modules"]` trong `config.py`.

In [6]:
!python train.py

PROFILE          : demo
Device           : cuda
QLoRA (4-bit)    : False
Batch size       : 2  x grad_accum 2  = effective 4
Epochs           : 2
Max train samples: 200
processor_config.json: 100% 68.0/68.0 [00:00<00:00, 374kB/s]
config.json: 100% 1.03k/1.03k [00:00<00:00, 3.74MB/s]
model.safetensors.index.json: 100% 122k/122k [00:00<00:00, 216MB/s]
Fetching 2 files: 100% 2/2 [02:47<00:00, 83.63s/it] 
Download complete: 100% 15.0G/15.0G [02:47<00:00, 89.5MB/s]                
Loading weights: 100% 1247/1247 [00:59<00:00, 20.95it/s]
generation_config.json: 100% 141/141 [00:00<00:00, 826kB/s]
trainable params: 473,088 || all params: 3,745,234,944 || trainable%: 0.012631730908040999
Số mẫu train: 118  |  Số batch/epoch: 59
Epoch 1/2  loss = 7.3485
  ↳ đã lưu checkpoint: ./blip2_lora_demo/epoch_1
Epoch 2/2  loss = 7.3884
  ↳ đã lưu checkpoint: ./blip2_lora_demo/epoch_2

✓ Hoàn tất. Adapter cuối lưu tại: ./blip2_lora_demo/final
  (file adapter_model.safetensors chỉ vài chục MB — đúng như bả

## 5. Inference — so sánh trước/sau fine-tune

Chọn 1 mẫu từ tập demo, chạy cả BLIP-2 gốc lẫn bản đã fine-tune để xem khác biệt.

In [7]:
import json
sample = json.load(open('data/train_converted.json'))[0]
img = f"data/vizwiz/train/{sample['image']}"
print(f"Question: {sample['question']}")
print(f"Ground truth: {sample['answer']}\n")
!python infer.py --adapter ./blip2_lora_demo/final \
    --image "{img}" --question "{sample['question']}" --compare

Question: What's the name of this product?
Ground truth: basil leaves

Fetching 2 files: 100% 2/2 [00:00<00:00, 5555.37it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights: 100% 1247/1247 [00:58<00:00, 21.16it/s]
[BLIP-2 GỐC]      basil leaves
[SAU FINE-TUNE]   basil leaves
